In [1]:
import pandas as pd
from datetime import datetime
from feast import FeatureStore

In [2]:
DATA_PATH = "feature_store/data"

In [3]:
data_orig = pd.read_parquet(f"{DATA_PATH}/driver_stats.parquet")

In [4]:
data_orig.head()

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2024-10-17 12:07:08.228578+00:00,1001,1.000000,1.000000,1000,2024-10-17 12:07:08.228581
1,2024-10-02 11:00:00+00:00,1005,0.429879,0.194598,582,2024-10-17 11:30:07.072000
2,2024-10-02 12:00:00+00:00,1005,0.230119,0.642878,551,2024-10-17 11:30:07.072000
3,2024-10-02 13:00:00+00:00,1005,0.128600,0.674187,38,2024-10-17 11:30:07.072000
4,2024-10-02 14:00:00+00:00,1005,0.400603,0.473636,583,2024-10-17 11:30:07.072000


In [5]:
data = data_orig.copy()

In [6]:
data.dtypes

event_timestamp    datetime64[ns, UTC]
driver_id                        int64
conv_rate                      float32
acc_rate                       float32
avg_daily_trips                  int32
created                 datetime64[us]
dtype: object

In [7]:
data.isnull().sum()

event_timestamp    0
driver_id          0
conv_rate          0
acc_rate           0
avg_daily_trips    0
created            0
dtype: int64

In [8]:
store = FeatureStore(repo_path="feature_store")
store

/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/repo_config.py:383: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


FeatureStore(
    repo_path=PosixPath('feature_store'),
    config=RepoConfig(project='feature_store', project_description=None, provider='local', registry_config='data/registry.db', online_config={'type': 'sqlite', 'path': 'data/online_store.db'}, auth={'type': 'no_auth'}, offline_config='dask', batch_engine_config='local', feature_server=None, flags=None, repo_path=PosixPath('feature_store'), entity_key_serialization_version=2, coerce_tz_aware=True, materialization_config=MaterializationConfig(pull_latest_features=False), openlineage_config=None),
    registry=not loaded,
    provider=not loaded
)

In [9]:
print("Feature Views:")
for fv in store.list_feature_views():
    print(f"  {fv.name}: {[f.name for f in fv.features]}")

Feature Views:
  driver_activity_fv: ['avg_daily_trips']
  driver_conversion_fv: ['conv_rate', 'acc_rate']


In [10]:
print("On-Demand Feature Views:")
for odfv in store.list_on_demand_feature_views():
    print(f"  {odfv.name}: {[f.name for f in odfv.features]}")

On-Demand Feature Views:
  driver_rt_odfv: ['conv_rate_plus_val1', 'conv_rate_plus_val2', 'performance_score', 'trips_x_acc_rate']


In [11]:
print("Feature Services:")
for fs in store.list_feature_services():
    print(f"  {fs.name}")

Feature Services:
  driver_model_v1


In [12]:
entity_df = pd.DataFrame({
    "driver_id": [1001, 1002, 1003],
    "event_timestamp": [
        datetime(2021, 4, 12, 10, 59, 42),
        datetime(2021, 4, 12, 8,  12, 10),
        datetime(2021, 4, 12, 16, 40, 26),
    ],
    "val_to_add":   [1, 2, 3],
    "val_to_add_2": [10, 20, 30],
})

In [13]:
batch_training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_conversion_fv:conv_rate",
        "driver_conversion_fv:acc_rate",
        "driver_activity_fv:avg_daily_trips",
    ],
).to_df()

display(batch_training_df)

,driver_id,event_timestamp,val_to_add,val_to_add_2,conv_rate,acc_rate,avg_daily_trips
0,1001,2021-04-12 10:59:42+00:00,1,10,0.709758,0.692957,402
1,1002,2021-04-12 08:12:10+00:00,2,20,0.718295,0.584081,370
2,1003,2021-04-12 16:40:26+00:00,3,30,0.697411,0.197680,25


In [14]:
full_training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_conversion_fv:conv_rate",
        "driver_conversion_fv:acc_rate",
        "driver_activity_fv:avg_daily_trips",
        "driver_rt_odfv:conv_rate_plus_val1",
        "driver_rt_odfv:conv_rate_plus_val2",
        "driver_rt_odfv:performance_score",
        "driver_rt_odfv:trips_x_acc_rate",
    ],
).to_df()

display(full_training_df)

,driver_id,event_timestamp,val_to_add,val_to_add_2,conv_rate,acc_rate,avg_daily_trips,conv_rate_plus_val1,conv_rate_plus_val2,performance_score,trips_x_acc_rate
0,1001,2021-04-12 10:59:42+00:00,1,10,0.709758,0.692957,402,1.709758,10.709758,0.703038,278.568827
1,1002,2021-04-12 08:12:10+00:00,2,20,0.718295,0.584081,370,2.718295,20.718295,0.664610,216.110100
2,1003,2021-04-12 16:40:26+00:00,3,30,0.697411,0.197680,25,3.697411,30.697411,0.497519,4.942010


In [15]:
fs = store.get_feature_service("driver_model_v1")

service_training_df = store.get_historical_features(
    entity_df=entity_df,
    features=fs,
).to_df()

display(service_training_df)

,driver_id,event_timestamp,val_to_add,val_to_add_2,conv_rate,acc_rate,avg_daily_trips,conv_rate_plus_val1,conv_rate_plus_val2,performance_score,trips_x_acc_rate
0,1001,2021-04-12 10:59:42+00:00,1,10,0.709758,0.692957,402,1.709758,10.709758,0.703038,278.568827
1,1002,2021-04-12 08:12:10+00:00,2,20,0.718295,0.584081,370,2.718295,20.718295,0.664610,216.110100
2,1003,2021-04-12 16:40:26+00:00,3,30,0.697411,0.197680,25,3.697411,30.697411,0.497519,4.942010


In [18]:
store.materialize_incremental(
    end_date=datetime(2025, 12, 31, 23, 59, 59)
)

Materializing 2 feature views to 2025-12-31 23:59:59+00:00 into the sqlite online store.

driver_activity_fv from 2021-04-12 23:59:59+00:00 to 2025-12-31 23:59:59+00:00:
driver_conversion_fv from 2021-04-12 23:59:59+00:00 to 2025-12-31 23:59:59+00:00:


/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(
/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


In [19]:
online_conversion = store.get_online_features(
    features=[
        "driver_conversion_fv:conv_rate",
        "driver_conversion_fv:acc_rate",
    ],
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1002},
        {"driver_id": 1003},
    ],
).to_dict()

display(pd.DataFrame(online_conversion))

,driver_id,conv_rate,acc_rate
0,1001,1.000000,1.000000
1,1002,0.134940,0.982010
2,1003,0.934453,0.695821


In [20]:
online_activity = store.get_online_features(
    features=[
        "driver_activity_fv:avg_daily_trips",
    ],
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1002},
        {"driver_id": 1003},
    ],
).to_dict()

display(pd.DataFrame(online_activity))

/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


,driver_id,avg_daily_trips
0,1001,1000
1,1002,696
2,1003,763


In [21]:
online_rt = store.get_online_features(
    features=[
        "driver_conversion_fv:conv_rate",
        "driver_conversion_fv:acc_rate",
        "driver_activity_fv:avg_daily_trips",
        "driver_rt_odfv:conv_rate_plus_val1",
        "driver_rt_odfv:conv_rate_plus_val2",
        "driver_rt_odfv:performance_score",
        "driver_rt_odfv:trips_x_acc_rate",
    ],
    entity_rows=[
        {"driver_id": 1001, "val_to_add": 1, "val_to_add_2": 10},
        {"driver_id": 1002, "val_to_add": 2, "val_to_add_2": 20},
        {"driver_id": 1003, "val_to_add": 3, "val_to_add_2": 30},
    ],
).to_dict()

display(pd.DataFrame(online_rt))

/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


,driver_id,conv_rate,acc_rate,avg_daily_trips,conv_rate_plus_val1,conv_rate_plus_val2,performance_score,trips_x_acc_rate
0,1001,1.000000,1.000000,1000,2.000000,11.000000,1.000000,1000.000000
1,1002,0.134940,0.982010,696,2.134940,20.134940,0.473768,683.479214
2,1003,0.934453,0.695821,763,3.934453,30.934453,0.839000,530.911049


In [22]:
fs = store.get_feature_service("driver_model_v1")

online_service = store.get_online_features(
    features=fs,
    entity_rows=[
        {"driver_id": 1001, "val_to_add": 1, "val_to_add_2": 10},
        {"driver_id": 1002, "val_to_add": 2, "val_to_add_2": 20},
    ],
).to_dict()

display(pd.DataFrame(online_service))

/Users/sofasabalova/miniconda3/envs/ml-global/lib/python3.11/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


,driver_id,conv_rate,acc_rate,avg_daily_trips,conv_rate_plus_val1,conv_rate_plus_val2,performance_score,trips_x_acc_rate
0,1001,1.00000,1.00000,1000,2.00000,11.00000,1.000000,1000.000000
1,1002,0.13494,0.98201,696,2.13494,20.13494,0.473768,683.479214
